# Survival Analysis of Unemployment Duration
### ECON 148 — Data Science for Economics

In this notebook we analyze how long displaced workers remain unemployed before finding a new job. Along the way we will:

- Motivate why ordinary regression fails for duration data
- Understand censoring and the survival function S(t)
- Estimate and plot Kaplan-Meier survival curves
- Compare survival across groups (education, recession period)
- Fit a Cox proportional hazards model to estimate covariate effects

**Data**: CPS Displaced Worker Supplement (IPUMS), biennial waves 2002–2022. Each row is a worker who lost their job due to a layoff or plant closure. The outcome of interest is re-employment.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [ ]:
df = pd.read_csv('displacement_survival.csv')

# Restore ordered categorical
edu_order = ['Less than HS', 'HS diploma', 'Some college', "Bachelor's+"]
df['education'] = pd.Categorical(df['education'], categories=edu_order, ordered=True)

print(f'Observations: {len(df):,}')
print(f'Re-employed (event=1): {df["event"].sum():,} ({df["event"].mean():.1%})')
print(f'Censored    (event=0): {(1-df["event"]).sum():,} ({(1-df["event"]).mean():.1%})')
df.head()

### The two key columns

Every survival analysis dataset needs exactly two things:

| column | meaning |
|---|---|
| `duration` | weeks from job loss until re-employment **or** the survey date, whichever came first |
| `event` | 1 if re-employed by survey date; 0 if still searching (right-censored) |

Workers with `event=0` are **censored** — we know they survived at least `duration` weeks without finding work, but we don't know what happened after the survey.

---
## 1. Why not just use OLS?

Before fitting anything, let's see what goes wrong if we ignore censoring and treat `duration` as a plain outcome.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: distribution of duration — heavily right-skewed
axes[0].hist(df['duration'], bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Weeks unemployed')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of spell duration')

# Right: naive comparison — mean duration for re-employed vs censored
# This is WRONG but illustrates the problem
means = df.groupby('event')['duration'].mean()
labels = ['Censored\n(still searching)', 'Re-employed']
colors = ['#e07b54', '#4c8cbf']
axes[1].bar(labels, means.values, color=colors, width=0.5)
axes[1].set_ylabel('Mean weeks')
axes[1].set_title('Naive mean duration by outcome\n(misleading — do not interpret causally)')
for i, v in enumerate(means.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('Notice: censored spells have SHORTER mean duration than re-employed spells.')
print('This is the length-biased sampling problem — people censored early look like short spells.')
print('OLS on duration would give biased estimates of both the mean and any covariate effects.')

**Three reasons OLS fails here:**

1. **Censoring**: We can't include censored observations sensibly in a regression — dropping them biases the sample toward longer spells; coding them at their censoring time biases toward shorter ones.
2. **Skewness**: Spell durations are right-skewed and non-negative. Log-transforming helps but doesn't solve the censoring problem.
3. **Time-varying hazard**: We often care about *when* the risk of re-employment is highest (e.g., does it spike right before UI benefits expire?). OLS averages over time and obscures this.

---
## 2. The Kaplan-Meier estimator

The **survival function** S(t) = P(T > t) is the probability that a worker has *not* yet found a job by week t.

The Kaplan-Meier estimator computes S(t) nonparametrically:

$$\hat{S}(t) = \prod_{t_i \leq t} \left(1 - \frac{d_i}{n_i}\right)$$

where $d_i$ is the number of events (re-employments) at time $t_i$ and $n_i$ is the number of workers still at risk (unemployed and uncensored) just before $t_i$. Censored observations exit the risk set without causing a drop in the curve.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

kmf = KaplanMeierFitter()
kmf.fit(df['duration'], event_observed=df['event'], label='All displaced workers')
kmf.plot_survival_function(ax=ax, ci_show=True, at_risk_counts=True)

# Mark median survival
median_surv = kmf.median_survival_time_
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline(median_surv, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax.annotate(f'Median: {median_surv:.0f} weeks',
            xy=(median_surv, 0.5),
            xytext=(median_surv + 3, 0.55),
            fontsize=9, color='gray')

ax.set_xlabel('Weeks since job loss')
ax.set_ylabel('S(t)  —  probability still unemployed')
ax.set_title('Kaplan-Meier survival function: unemployment spells')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'Median survival time: {median_surv:.0f} weeks')
print(f'S(4 weeks):  {kmf.survival_function_at_times([4]).values[0]:.3f}')
print(f'S(13 weeks): {kmf.survival_function_at_times([13]).values[0]:.3f}')
print(f'S(26 weeks): {kmf.survival_function_at_times([26]).values[0]:.3f}')
print(f'S(52 weeks): {kmf.survival_function_at_times([52]).values[0]:.3f}')

**Reading the curve**: The y-axis is the probability of *still being unemployed* at each week. S(t) = 0.5 at the median — half of displaced workers are still searching at that point. Each vertical drop is one or more re-employment events. The shaded band is a 95% confidence interval (Greenwood's formula).

---
## 3. Comparing groups: education

Does education affect the speed of re-employment? We plot separate K-M curves for each education level.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#c0392b', '#e67e22', '#2980b9', '#27ae60']

# Ensure numeric inputs for lifelines
df_plot = df.copy()
df_plot['duration'] = pd.to_numeric(df_plot['duration'], errors='coerce')
df_plot['event'] = pd.to_numeric(df_plot['event'], errors='coerce')

medians = {}
for color, edu in zip(colors, edu_order):
    sub = df_plot.loc[df_plot['education'] == edu, ['duration', 'event']].dropna()

    # Skip empty groups to avoid ValueError: Empty array/Series passed in.
    if sub.empty:
        print(f"Skipping {edu}: no valid rows")
        continue

    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=sub['duration'].astype(float),
        event_observed=sub['event'].astype(int),
        label=edu
    )
    kmf.plot_survival_function(ax=ax, ci_show=False, color=color)
    medians[edu] = kmf.median_survival_time_

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
ax.set_xlabel('Weeks since job loss')
ax.set_ylabel('S(t)  —  probability still unemployed')
ax.set_title('Kaplan-Meier curves by education level')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('Median unemployment duration by education (weeks):')
for edu, m in medians.items():
    print(f'  {edu:<18}: {m:.0f}')

In [ ]:
# Log-rank test: are the survival curves statistically different?
result = multivariate_logrank_test(
    df['duration'],
    df['education'],
    event_observed=df['event']
)
result.print_summary()
print(f'\nInterpretation: p = {result.p_value:.4f}')
print('The log-rank test asks: could all four curves have come from the same underlying distribution?')

---
## 4. The Great Recession vs. other periods

The 2008–2010 waves captured workers displaced during the worst labor market downturn since the Depression. How did displacement during the Great Recession affect re-employment duration compared to other periods?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label, mask, color in [
    ('Great Recession (2008–2010)', df['great_recession'] == 1, '#c0392b'),
    ('Other years',                 df['great_recession'] == 0, '#2980b9'),
]:
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, 'duration'],
            event_observed=df.loc[mask, 'event'],
            label=label)
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
ax.set_xlabel('Weeks since job loss')
ax.set_ylabel('S(t)  —  probability still unemployed')
ax.set_title('Kaplan-Meier curves: Great Recession vs. other periods')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Two-sample log-rank test
results = logrank_test(
    df.loc[df['great_recession']==1, 'duration'],
    df.loc[df['great_recession']==0, 'duration'],
    event_observed_A=df.loc[df['great_recession']==1, 'event'],
    event_observed_B=df.loc[df['great_recession']==0, 'event'],
)
results.print_summary()

---
## 5. The hazard function

The survival function S(t) tells us the *cumulative* probability of still being unemployed. The **hazard function** h(t) tells us something different: the *instantaneous rate* of re-employment at time t, given that the worker has been unemployed up to t.

$$h(t) = \lim_{\Delta t \to 0} \frac{P(t \leq T < t + \Delta t \mid T \geq t)}{\Delta t}$$

Think of h(t) as the "exit rate" from unemployment at each moment. A high hazard means workers are finding jobs quickly. A low hazard means they are stuck.

In [ ]:
from lifelines import NelsonAalenFitter

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#c0392b', '#2980b9']
labels = ['Great Recession (2008–2010)', 'Other years']
masks  = [df['great_recession'] == 1, df['great_recession'] == 0]

for ax, (color, label, mask) in zip(axes, zip(colors, labels, masks)):
    naf = NelsonAalenFitter()
    naf.fit(df.loc[mask, 'duration'],
            event_observed=df.loc[mask, 'event'],
            label=label)
    # Plot smoothed hazard
    naf.plot_hazard(bandwidth=4, ax=ax, color=color)
    ax.set_xlabel('Weeks since job loss')
    ax.set_ylabel('Hazard rate h(t)')
    ax.set_title(f'Smoothed hazard — {label}')

plt.tight_layout()
plt.show()

print('The hazard rate shows the re-employment intensity at each duration.')
print('A declining hazard suggests negative duration dependence: the longer you are unemployed,')
print('the harder it becomes to find a job (skill depreciation, stigma, discouragement).')

---
## 6. Cox proportional hazards model

K-M curves compare groups but can only handle one variable at a time. To estimate the *partial* effect of education controlling for age, sex, and recession period, we use the **Cox model**:

$$h(t \mid X) = h_0(t) \cdot \exp(\beta_1 X_1 + \beta_2 X_2 + \cdots)$$

Key features:
- $h_0(t)$ is the **baseline hazard** — the hazard for a reference individual. It is left unspecified (nonparametric), which is what makes the model "semi-parametric".
- The covariates shift the baseline hazard *multiplicatively* and *proportionally* at all time points.
- $\exp(\beta_j)$ is the **hazard ratio**: the multiplicative change in re-employment rate per unit increase in $X_j$, holding other covariates fixed.
- A hazard ratio > 1 means a covariate *increases* the exit rate from unemployment (faster re-employment).

In [ ]:
# Prepare model dataframe — Cox needs numeric covariates
model_df = df[['duration', 'event', 'female', 'age', 'education',
               'married', 'great_recession']].copy()

# Encode education as dummies (Less than HS is reference)
model_df = pd.get_dummies(model_df, columns=['education'], drop_first=True)

# Rename for readability
rename = {
    'education_HS diploma':    'educ_HS',
    'education_Some college':  'educ_some_college',
    "education_Bachelor's+":   'educ_bachelors_plus',
}
model_df.rename(columns=rename, inplace=True)

# Drop educ_bachelors_plus — constant column in this dataset (no variation), causes singular Hessian
model_df = model_df.drop(columns=['educ_bachelors_plus'], errors='ignore')

model_df.head(3)

In [ ]:
cph = CoxPHFitter()
cph.fit(model_df, duration_col='duration', event_col='event')
cph.print_summary(decimals=3)

In [ ]:
# Forest plot of hazard ratios
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('log(Hazard Ratio)  —  positive = faster re-employment')
ax.set_title('Cox model: covariate effects on re-employment hazard')
plt.tight_layout()
plt.show()

print('Hazard ratios (exp of coefficients):')
hrs = np.exp(cph.params_)
for name, hr in hrs.items():
    print(f'  {name:<25}: {hr:.3f}')

**Interpreting hazard ratios:**

- `great_recession = 0.XX` — displaced workers in the Great Recession had a ~XX% *lower* re-employment hazard at every duration, controlling for demographics. This captures the aggregate demand shock.
- `educ_HS` / `educ_some_college` — hazard ratios relative to workers without a high school diploma (the reference group), holding age and recession period constant.
- `female = X.XX` — interpret carefully: this reflects *observed* re-employment rates and may conflate labor force attachment decisions with job search success.

---
## 7. Discussion questions

1. **Negative duration dependence**: Does the hazard function suggest that the longer a worker has been unemployed, the harder it becomes to find a new job? What mechanisms (human capital depreciation, employer signaling, discouragement) could explain this?

2. **Education and the recession**: Do the education group K-M curves converge or diverge during the Great Recession? What does this tell us about whether education acts as an insurance mechanism against long-term unemployment?

3. **Censoring and selection**: Workers who are still unemployed at the survey date are censored. Are they a random subset of all unemployed workers, or might they be systematically different (e.g., older, lower-educated)? What does this imply for our estimates?

4. **Proportional hazards**: If the PH assumption is violated for any covariate, what are our options? (Stratified Cox model, time-varying coefficients, parametric models.)

5. **Policy implications**: The UI system in the US provides benefits for up to 26 weeks (extended to 99 weeks during the Great Recession). Do you see any evidence of a spike in re-employment hazard around benefit exhaustion? How would you test this more formally?